In [1]:
from pathlib import Path

# The dataset files are stored inside the "nl-en" subfolder
data_folder = Path("../data")

# Define the relative paths to the English and Dutch files
english_file = data_folder / "europarl-v7.nl-en.en"
dutch_file = data_folder / "europarl-v7.nl-en.nl"

# Check that both files can be found
print(english_file.exists())
print(dutch_file.exists())

True
True


In [2]:
with english_file.open("r", encoding="utf-8") as english_data:
    english_line_count = sum(1 for line in english_data)

with dutch_file.open("r", encoding="utf-8") as dutch_data:
    dutch_line_count = sum(1 for line in dutch_data)

# Probably unnecessary, just make sure both files have the exact same number of lines
print("Number of English lines:", english_line_count)
print("Number of Dutch lines:  ", dutch_line_count)

Number of English lines: 1997775
Number of Dutch lines:   1997775


In [3]:
import random
# Use 10% of the corpus
sample_fraction = 0.1

# Calculate the nearest possible whole number of sentence pairs
sample_size = round(english_line_count * sample_fraction)

# Use a fixed seed so the same sample can be reproduced
random_generator = random.Random(42)

# Randomly select indices without selecting the same pair twice
sampled_line_indices = set(random_generator.sample(range(english_line_count), sample_size))

print("Total aligned pairs:", english_line_count)
print("Selected pairs:", len(sampled_line_indices))

Total aligned pairs: 1997775
Selected pairs: 199778


In [4]:
def preprocess_pair(english_sentence, dutch_sentence):
    """
    Preprocess one aligned English-Dutch sentence pair.

    Returns:
        A cleaned (English, Dutch) tuple if the pair is valid.
        None if either sentence is empty or begins with an XML tag.
    """
    # Remove whitespaces and replace repeated whitespace with single one
    english_sentence = " ".join(english_sentence.split())
    dutch_sentence = " ".join(dutch_sentence.split())

    # Remove the pair if either sentence is empty
    if not english_sentence or not dutch_sentence:
        return None

    # Remove if either sentence begins with an XML tag.
    if english_sentence.startswith("<") or dutch_sentence.startswith("<"):
        return None

    # Convert to lowercase
    english_sentence = english_sentence.lower()
    dutch_sentence = dutch_sentence.lower()

    return english_sentence, dutch_sentence

In [5]:
preprocessed_pairs = []

line_index = 0
# The reason why we are randomly selecting sentence pair is that we are uncertain about if the original dataset in order creates bias or not
with open(english_file, "r", encoding="utf-8") as english_data:
    with open(dutch_file, "r", encoding="utf-8") as dutch_data:
        # Read one English and one Dutch line 
        for english_line, dutch_line in zip(english_data, dutch_data):
            # Check whether this line belongs to the random sample
            if line_index in sampled_line_indices:
                # Apply the preprocessing function.
                processed_pair = preprocess_pair(english_line, dutch_line)
                # Keep the pair if it was not removed
                if processed_pair is not None:
                    preprocessed_pairs.append(processed_pair)
            # Move to the next line number.
            line_index = line_index + 1

In [6]:
# Create the preprocessed output file
english_output_file = Path("../data/preprocessed_10_percent.en")
dutch_output_file = Path("../data/preprocessed_10_percent.nl")

# Open both output files
with open(english_output_file, "w", encoding="utf-8") as english_output:
    with open(dutch_output_file, "w", encoding="utf-8") as dutch_output:
        # Write each aligned pair to the two files
        for english_sentence, dutch_sentence in preprocessed_pairs:

            english_output.write(english_sentence + "\n")
            dutch_output.write(dutch_sentence + "\n")

print("Preprocessed files saved successfully.")

Preprocessed files saved successfully.
